# Task 3: Agentic Financial Research with LangGraph
This notebook runs a single autonomous research agent and a two-agent analyst/writer workflow with mandatory clarification. Both reuse Task 1. The LLM chooses tools from observations; graph edges enforce the decision loop and role handoff, not a tool order.

Run all cells in order. Outputs are empty by design. Real execution requires Yahoo/Groq/search availability and a Groq JSON-capable model. Do not paste keys into notebook source. Test demonstrations are separate offline pytest runs, not fabricated research results.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Randimal/CDAZZDEV-MLE-Poshitha-Malagala.git"
cwd = Path.cwd()
repo_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "task3_agentic" / "single_agent.py").exists()
    ),
    None,
)
if repo_root is None:
    if not REPOSITORY_URL.startswith("https://github.com/"):
        raise ValueError("Set REPOSITORY_URL and rerun this cell.")
    repo_root = cwd / "CDAZZDEV-MLE-Poshitha-Malagala"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--", REPOSITORY_URL, str(repo_root)], check=True
        )
    if not (repo_root / "task3_agentic" / "single_agent.py").exists():
        raise ValueError("Checkout does not contain Task 3.")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repo_root / "requirements.txt"),
    ],
    check=True,
)
sys.path.insert(0, str(repo_root))

## Configuration and secure credentials
Load GROQ_API_KEY and GROQ_MODEL from the environment, Colab userdata, or hidden key input. Set a model supporting JSON object mode. Task 3 uses a larger completion budget for structured reports. Cache dates use UTC; price observations retain their own provider timestamps.

In [ ]:
import getpass
import os
from datetime import datetime, timezone

import pandas as pd
from IPython.display import JSON, display

from task1_financial.llm import GroqClient, LLMConfigurationError
from task3_agentic.memory import PersistentMemory, answer_followup
from task3_agentic.multi_agent import TwoAgentResearch
from task3_agentic.prompts import QUESTION
from task3_agentic.runtime import AgentRuntime
from task3_agentic.single_agent import SingleResearchAgent
from task3_agentic.tools import FinancialTools, ToolExecutor
from task3_agentic.tracing import ToolTracer

for name in ("GROQ_API_KEY", "GROQ_MODEL"):
    if not os.environ.get(name, "").strip():
        try:
            from google.colab import userdata

            value = userdata.get(name)
            if value:
                os.environ[name] = value.strip()
        except Exception:
            pass
if not os.environ.get("GROQ_API_KEY", "").strip():
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key (hidden): ").strip()
if not os.environ.get("GROQ_MODEL", "").strip():
    os.environ["GROQ_MODEL"] = input("Groq JSON-capable model identifier: ").strip()


TICKER = "NVDA"
RUN_DATE = datetime.now(timezone.utc).date()
QUERY = QUESTION.format(ticker=TICKER)
TRACE_PATH = repo_root / "task3_agentic" / "logs" / "agent_trace.jsonl"
MEMORY_DIR = repo_root / "task3_agentic" / "memory"
client = None
try:
    client = GroqClient(max_completion_tokens=1800)
except LLMConfigurationError:
    print("Configure GROQ_API_KEY and GROQ_MODEL, then rerun.")
except Exception:
    print("Client initialization unavailable; verify settings without sharing secrets.")


def make_runtime():
    tools = FinancialTools(TICKER, client)
    return AgentRuntime(client, ToolExecutor(tools, ToolTracer(TRACE_PATH)))


memory = PersistentMemory(MEMORY_DIR)

## Five callable tools
This explicit showcase is independent of autonomous agent execution. The agents below use fresh sessions and choose their own order. Every showcase invocation passes through the traced dispatcher. A failure is an observation, not a notebook crash. Sentiment uses only actually retrieved headlines.

In [ ]:
demo_observations = []
if client is not None:
    demo = make_runtime().executor
    for name, arguments in (
        ("get_price_data", {"ticker": TICKER, "period": "2y"}),
        ("get_news", {"ticker": TICKER, "n": 10}),
        ("calculate_volatility", {"ticker": TICKER, "window": 60}),
        (
            "web_search",
            {"query": f"{TICKER} current share price risks official filings"},
        ),
    ):
        observation = demo.invoke("researcher", name, arguments)
        demo_observations.append(observation)
        print(name, "success:", observation.success)
        display(JSON(observation.model_dump(mode="json")))
    headlines = [vars(item) for item in demo.tools.known_headlines.values()]
    observation = demo.invoke("researcher", "llm_sentiment", {"headlines": headlines})
    demo_observations.append(observation)
    display(JSON(observation.model_dump(mode="json")))
else:
    print("Configure Groq before the live tool showcase.")

## Single autonomous research agent
Fresh execution bypasses persistent reuse for this first demonstration. Tool failures and empty results return to the LLM for another decision. The decision budget bounds cost; if no grounded report is possible, inspect the error/trace instead of interpreting invented output.

In [ ]:
single = None
if client is not None:
    single_runtime = make_runtime()
    single = SingleResearchAgent(single_runtime, memory).run(
        QUERY, as_of=RUN_DATE, use_cache=False
    )
    display(
        JSON(
            single.report.model_dump(mode="json")
            if single.report
            else {"unavailable": single.error}
        )
    )

In [ ]:
def show_trace(run):
    display(
        pd.DataFrame(
            [
                {
                    "timestamp": item.timestamp,
                    "role": item.role,
                    "event": item.event,
                    "content": item.content,
                }
                for item in run.trace
            ]
        )
    )
    # Full sanitized LLM messages, decisions, observations and handoffs.
    display(JSON([item.model_dump(mode="json") for item in run.trace], expanded=True))


if single is not None:
    show_trace(single)
    cycle = [
        item.event
        for item in single.trace
        if item.event in {"decision", "tool_call", "observation", "replan"}
    ]
    print("Observed decision/tool/observation/replan cycle:", cycle)
    # Inspect the subsequent decision's reason to see how the model replanned.
    decisions = [item.content for item in single.trace if item.event == "decision"]
    display(JSON(decisions, expanded=True))

## Two-agent workflow: mandatory review and clarification
Agent A can use only price, volatility and sentiment. Agent B can use only news and web search. A passes a Pydantic quantitative brief; B must produce one structured clarification request, A must respond, and B must incorporate that response. Code rejects invented numerical handoff values and missing clarification incorporation.

In [ ]:
multi = None
if client is not None:
    multi_runtime = make_runtime()
    multi = TwoAgentResearch(multi_runtime, memory).run(
        QUERY, as_of=RUN_DATE, use_cache=False
    )
    print("Agent A quantitative brief")
    display(
        JSON(multi.brief.model_dump() if multi.brief else {"unavailable": multi.error})
    )
    print("Agent B critique request")
    display(
        JSON(
            multi.critique_request.model_dump()
            if multi.critique_request
            else {"unavailable": True}
        )
    )
    print("Agent A clarification response")
    display(
        JSON(
            multi.clarification.model_dump()
            if multi.clarification
            else {"unavailable": True}
        )
    )
    print("Agent B final report")
    display(
        JSON(
            multi.report.model_dump(mode="json")
            if multi.report
            else {"unavailable": multi.error}
        )
    )
    show_trace(multi)
    if multi.report is not None:
        assert multi.critique_request is not None and multi.clarification is not None
        assert multi.report.clarification_used == multi.clarification.answer
        print("Mandatory critique loop completed and incorporated.")

## Short-term memory: no repeat price/volatility fetch
The follow-up helper receives saved state and has no access to a tool executor. It may make an LLM call, but cannot refetch data. Tool-call counters and trace-file size verify that property.

In [ ]:
if multi is not None and multi.report is not None:
    before = multi_runtime.executor.counter
    trace_bytes = TRACE_PATH.stat().st_size if TRACE_PATH.exists() else 0
    followup = answer_followup(
        multi,
        "What quantitative evidence supports the hedge, and what are its limitations?",
        client,
    )
    display(JSON(followup.model_dump() if followup else {"unavailable": True}))
    assert multi_runtime.executor.counter == before
    assert TRACE_PATH.stat().st_size == trace_bytes
    print("Follow-up answered from session state without another data-tool call.")

## Persistent memory: first saved run, then a new-session cache hit
A valid report from the first run is saved to TICKER_UTC-DATE.json. Single and multi results occupy separate workflow slots so a cached single report cannot bypass the required two-agent critique. Cache bypass is explicit with use_cache=False. A new client-free session below proves the second run does not require API calls.

In [ ]:
if multi is not None and multi.report is not None:
    cache_path = memory.path(TICKER, RUN_DATE)
    print(
        "First run saved:",
        cache_path.relative_to(repo_root),
        "exists:",
        cache_path.exists(),
    )

    class NoCallsClient:
        def complete(self, system, user):
            raise AssertionError("A cache hit must not call an LLM")

    cached_client = NoCallsClient()
    cached_runtime = AgentRuntime(
        cached_client,
        ToolExecutor(FinancialTools(TICKER, cached_client), ToolTracer(TRACE_PATH)),
    )
    before_bytes = TRACE_PATH.stat().st_size
    second = TwoAgentResearch(cached_runtime, memory).run(QUERY, as_of=RUN_DATE)
    assert second.cached and cached_runtime.executor.counter == 0
    assert TRACE_PATH.stat().st_size == before_bytes
    print("Second run: persistent cache hit; zero tools and zero LLM calls.")
    display(JSON(second.report.model_dump(mode="json")))
    display(JSON(second.trace[-1].model_dump(mode="json")))
else:
    print(
        "No completed report: nothing was cached. Retry after resolving availability."
    )

## Tool observability
Every dispatcher invocation, including permission failures and session cache hits, appends a redacted record. Output is limited to 200 characters; full model-facing observations remain in the session trace. No API keys or raw exception bodies are logged. Logs/cache are ignored by Git.

In [ ]:
import json

if TRACE_PATH.exists():
    rows = [
        json.loads(line) for line in TRACE_PATH.read_text(encoding="utf-8").splitlines()
    ]
    display(pd.DataFrame(rows))
    assert all(len(row["output"]) <= 200 for row in rows)
    print("JSONL tool-call count:", len(rows))
else:
    print("No tool trace yet; execute the live workflow cells first.")

## Limitations and submission checks
Review the report's actual sources and evidence IDs. Search snippets can be stale or misleading; OHLCV does not establish balance-sheet health. Historical volatility is not a forecast, and an options hedge cannot be priced without an options chain. Structural/numeric grounding does not validate all qualitative LLM claims. Same-day cache results can become stale; use a fresh session and bypass persistent caching to refresh. The fixed stage handoff implements the mandatory critique, while tool order within each stage is autonomous. Task 2 remains pending.

The live trace must visibly show at least two decisions separated by a tool observation/replan to demonstrate the cycle. If provider failures prevent this, rerun in Colab; do not claim live autonomous performance from the mocked tests alone.